# The covariance kernel: general theory and scope

This notebook derives the **general conditions** under which a 1D covariance kernel can recover nonlinear functionals of the $(p, m, j)$ joint from the TM's 1D ergodic, classifies the key HAFiscal calculations by whether they need a kernel, and demonstrates the welfare kernel alongside the marginal-utility kernel.

**Structure:** (1) When does a kernel exist? (2) Classification of HAFiscal calculations. (3) Setup. (4) Marginal utility and welfare kernels. (5) Why the Lorenz curve is harder. (6) Summary table.

## 1 -- When does a kernel exist?

### The problem

The TM tracks $\pi(m, j)$ -- the 1D ergodic distribution of normalized market resources and micro-state. Permanent income $p$ is not a TM state. For any functional $\mathbb{E}[g(p, m, j)]$, the TM must approximate the joint $f(p, m, j)$ from the marginals.

### The factorization via $p_{t-1}$

HARK's transition maps the end-of-period $(t{-}1)$ state to beginning-of-period $t$:

$$p_t = p_{t-1}\,\Phi_t, \qquad m_t = \frac{R\,a_{t-1}}{\Phi_t} + \theta_t, \qquad \Phi_t = G_{j_t}\,\Psi_t$$

where $a_{t-1} = m_{t-1} - c(m_{t-1}, j_{t-1})$ is savings, and $(\Psi_t, \theta_t, j_t)$ are fresh shocks.

**Key property (BST balanced growth):** In the ergodic, $p_{t-1}$ is approximately independent of $(a_{t-1}, j_{t-1})$. The $\psi$-channel creates $\text{Corr}(p, a) \approx -0.006$, but this is small.

### General kernel theorem (approximate)

Suppose $g(p_t, m_t, j_t)$ can be written as $g = h(p_{t-1}) \cdot \phi(a_{t-1}, \Phi_t, \theta_t, j_t)$ for some function $h$ of $p_{t-1}$ alone and some function $\phi$ of savings and fresh shocks. Then:

$$\mathbb{E}[g] \;\approx\; \mathbb{E}[h(p)] \;\times\; \sum_{m,j}\pi(m,j)\;\kappa_g(a(m,j),\;j)$$

where the **kernel** is $\kappa_g(a, j) = \sum_{j'} P(j \!\to\! j') \sum_{\Psi,\theta} \text{Pr}(\Psi,\theta|j') \; \phi(a, G_{j'}\Psi, \theta, j')$.

### When does this factorization apply?

It applies whenever $g(p_t, m_t, j_t)$ is **multiplicatively separable in $p$** after substituting $p_t = p_{t-1}\Phi_t$. Specifically, whenever $g = p^k \cdot f(m, j, \theta)$ for some power $k$, we get $h(p_{t-1}) = p_{t-1}^k$ and $\phi = \Phi^k \cdot f(Ra/\Phi + \theta, j', \theta)$.

## 2 -- Classification of HAFiscal calculations

### Case A: $p$-linear aggregates -- no kernel needed

$g = p \cdot f(m, j)$. Examples: `AggCons`, `AggIncome`, NPVs, multipliers, IRFs.

The Harmenberg neutral-measure factorization is **exact**: $\mathbb{E}[p \cdot f] = \mathbb{E}[p] \cdot \mathbb{E}_Q[f]$. **No kernel needed.**

### Case B: Multiplicatively separable in $p$ -- kernel applies

$g = p^k \cdot f(m, j, \theta)$ with $k \neq 1$. Examples:

| Functional | $k$ | Where used |
|-----------|-----|-----------|
| Marginal utility $u'(c) = (pX)^{-\rho}$ | $-\rho$ | `Welfare.py`, marginal welfare weights |
| CRRA welfare $u(c) = (pX)^{1-\rho}/(1{-}\rho)$ | $1{-}\rho$ | `Welfare.py` |
| Policy welfare impact $\Delta u$ | $1{-}\rho$ | Policy welfare comparison |

The kernel $\kappa_g(a, j) = \sum_{j'} P \sum_{\Psi,\theta} \text{Pr} \cdot \Phi^k \cdot f(Ra/\Phi + \theta, j', \theta)$ resolves the $\theta$--$m$ coupling for all of these.

### Case C: Distributional functionals -- kernel doesn't directly apply

Lorenz shares, Gini, quantiles require the **full CDF** of level wealth $W = p \cdot a$, not just $\mathbb{E}[p^k \cdot f]$. The CDF $F_W(w) = \mathbb{E}[\mathbf{1}(p \cdot a \leq w)]$ is not multiplicatively separable. But the product-measure approximation ($p \perp a$) gives a good CDF because $\text{Corr}(p, a) \approx -0.006$.

### Case D: Income-dependent policy (check phase-out)

$\text{check}(p)/p$ is nonlinear in $p$ and enters the consumption function argument. Not multiplicatively separable. Requires **$p$-buckets** (already in `_compute_check_buckets`).

## 3 -- Setup

In [1]:
import os, sys, warnings, contextlib, io
from copy import deepcopy
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm as _norm

%matplotlib inline
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"] = 11

_here = os.path.abspath(os.getcwd())
_FPC = None
for _p in [os.path.join(_here, "FromPandemicCode"), os.path.join(os.path.dirname(_here), "FromPandemicCode"), _here]:
    if os.path.isfile(os.path.join(_p, "verify_four_methods_agreement.py")):
        _FPC = _p; break
if _FPC is None:
    raise FileNotFoundError("Cannot find FromPandemicCode")
os.chdir(_FPC)
if _FPC not in sys.path: sys.path.insert(0, _FPC)
_HM = os.path.dirname(_FPC)
if _HM not in sys.path: sys.path.insert(0, _HM)

_argv = sys.argv[:]
sys.argv = [sys.argv[0], "1.01", "2.0", "0.7"]
import Parameters as _Params
_rp = _Params.return_parameters
def _rp_q(*a, **k):
    with contextlib.redirect_stdout(io.StringIO()): return _rp(*a, **k)
_Params.return_parameters = _rp_q

from tm_methods import build_tm_agg_fiscal, compute_pLvl_distribution, find_ergodic_distribution
from verify_four_methods_agreement import (
    HAS_DUAL, _build_single_type_economy, _fix_act_T_after_switch, _mc_burnin_tm_init,
)
from AggFiscalModel import DualAggFiscalType
sys.argv = _argv

EDUC = "highschool"; PERIODS = 100; WARMUP = 24; T_START = 50; N = 20_000; SEED = 42

econ, bd = _build_single_type_economy(educ=EDUC, agent_count=N, seed=SEED, act_T=PERIODS, agent_cls=DualAggFiscalType)
econ.solve()
agent = econ.agents[0]
CRRA = float(agent.CRRA); Splurge = float(agent.Splurge)
Rfree = float(agent.Rfree[0]); J = int(agent.num_base_MrkvStates)
sol = agent.solution[0]

_mc_burnin_tm_init(econ, warmup=WARMUP, neutral_measure=True, use_dual=True)
econ.save_state()
econ.switch_to_counterfactual_mode("base")
_fix_act_T_after_switch(econ, PERIODS)
econ.make_idiosyncratic_shock_histories()
mc = econ.run_experiment(**deepcopy(bd), Full_Output=True)
c_mc = mc["cLvl_all_splurge"]; p_mc = mc["pLvl_all"]; T = c_mc.shape[0]

tm_data = build_tm_agg_fiscal(agent, mCount=150, Cratio=1.0, neutral_measure=False)
erg = np.asarray(find_ergodic_distribution(tm_data["TranMatrix"]), float).ravel()
erg /= erg.sum()
mGrid = tm_data["dist_mGrid"]; M = len(mGrid)
erg2 = erg.reshape(J, M)
u_rate = float(1.0 - np.sum(erg2[0, :]))

c_nrm = np.empty((J, M)); a_nrm = np.empty((J, M))
for j in range(J):
    c_nrm[j, :] = sol.cFunc[j](mGrid, np.ones(M))
    a_nrm[j, :] = mGrid - c_nrm[j, :]
p_grid, p_w = compute_pLvl_distribution(agent, n_points=200, unemployment_rate=u_rate)

print(f"CRRA={CRRA}, Splurge={Splurge:.4f}. Setup complete.")

CRRA=2.0, Splurge=0.2461. Setup complete.


## 4 -- The general kernel: marginal utility ($k = -\rho$) and welfare ($k = 1-\rho$)

The only difference between the two kernels is the exponent $k$ on $\Phi$ and $X$:

| | Marginal utility | Welfare |
|-|-----------------|---------|
| $k$ | $-\rho$ | $1-\rho$ |
| Integrand | $\Phi^{-\rho} X^{-\rho}$ | $\Phi^{1-\rho} X^{1-\rho}/(1{-}\rho)$ |

We implement both with one function parameterized by `p_power`.

In [2]:
def general_kernel(agent, tm_data, ergodic_flat, p_power, CRRA,
                   n_p_points=200, include_splurge=True):
    """
    General covariance kernel for E[p^k * f(m, j, theta)].

    p_power : float -- exponent k on p. k=-rho for u', k=1-rho for welfare.
    Returns (estimate, E_pk, kappa_sum).
    """
    dist_mGrid = tm_data["dist_mGrid"]
    M = len(dist_mGrid)
    J = int(agent.num_base_MrkvStates)
    erg = np.asarray(ergodic_flat, float).ravel()
    erg = erg / erg.sum()
    erg2 = erg.reshape(J, M)
    u_rate = float(1.0 - np.sum(erg2[0, :]))

    p_grid, p_w = compute_pLvl_distribution(agent, n_points=n_p_points, unemployment_rate=u_rate)
    E_pk = float(np.sum(p_w * p_grid ** p_power))

    sol = agent.solution[0]
    Spl = float(agent.Splurge) if include_splurge else 0.0
    Rf = float(np.asarray(agent.Rfree).ravel()[0])
    PGF = np.asarray(agent.PermGroFac[0], float)[:J]
    MrkB = np.asarray(agent.MrkvArray[0], float)[:J, :J]
    ISD = agent.IncShkDstn[0]
    rho = float(CRRA)

    c_nrm = np.empty((J, M)); a_nrm = np.empty((J, M))
    for j in range(J):
        c_nrm[j, :] = sol.cFunc[j](dist_mGrid, np.ones(M))
        a_nrm[j, :] = dist_mGrid - c_nrm[j, :]

    kappa_sum = 0.0
    for j_old in range(J):
        for i in range(M):
            a_i = float(a_nrm[j_old, i])
            w_i = float(erg2[j_old, i])
            if w_i < 1e-20: continue
            k_i = 0.0
            for j_new in range(J):
                tp = float(MrkB[j_old, j_new])
                if tp < 1e-15: continue
                G_n = float(PGF[j_new])
                d_n = ISD[j_new]
                for s in range(len(d_n.pmv)):
                    psi_s = float(d_n.atoms[0][s])
                    th_s = float(d_n.atoms[1][s])
                    pr_s = float(d_n.pmv[s])
                    Phi = G_n * psi_s
                    m_next = (a_i * Rf / Phi + th_s) if a_i > 0 else th_s
                    c_next = float(sol.cFunc[j_new](m_next, 1.0))
                    X = max((1.0 - Spl) * c_next + Spl * th_s, 1e-16)
                    if abs(p_power - (1.0 - rho)) < 1e-10 and rho != 1.0:
                        integrand = Phi ** (1.0 - rho) * X ** (1.0 - rho) / (1.0 - rho)
                    elif abs(p_power - (-rho)) < 1e-10:
                        integrand = Phi ** (-rho) * X ** (-rho)
                    else:
                        integrand = Phi ** p_power * X ** p_power
                    k_i += tp * pr_s * integrand
            kappa_sum += w_i * k_i
    return E_pk * kappa_sum, E_pk, kappa_sum

print("Computing kernels ...")
mu_kernel, mu_Epk, mu_ksum = general_kernel(agent, tm_data, erg, -CRRA, CRRA)
w_kernel, w_Epk, w_ksum = general_kernel(agent, tm_data, erg, 1.0 - CRRA, CRRA)

mc_mu = float(np.mean([np.mean(np.maximum(c_mc[t], 1e-16)**(-CRRA)) for t in range(T_START, T)]))
mc_welfare = float(np.mean([np.mean(np.maximum(c_mc[t], 1e-16)**(1-CRRA)/(1-CRRA)) for t in range(T_START, T)]))

print("done.\n")
print(f"{'Functional':35s} {'Kernel':>14s} {'MC truth':>14s} {'Error':>8s}")
print("-" * 73)
print(f"{'E[u] = E[(pX)^{-rho}]':35s} {mu_kernel:14.8e} {mc_mu:14.8e} {(mu_kernel-mc_mu)/mc_mu*100:+7.3f}%")
print(f"{'E[u]  = E[(pX)^{1-rho}/(1-rho)]':35s} {w_kernel:14.8e} {mc_welfare:14.8e} {(w_kernel-mc_welfare)/mc_welfare*100:+7.3f}%")


Computing kernels ...


done.

Functional                                  Kernel       MC truth    Error
-------------------------------------------------------------------------
E[u] = E[(pX)^{-rho}]               1.28804247e-02 1.28620030e-02  +0.143%
E[u]  = E[(pX)^{1-rho}/(1-rho)]     -9.58654244e-02 -9.63670805e-02  -0.521%


## 5 -- Why the Lorenz curve is harder

The Lorenz curve at percentile $q$ is $L(q) = \int_0^{F_W^{-1}(q)} w\,dF_W(w)\;/\;\mathbb{E}[W]$. This requires the **quantile** $F_W^{-1}(q)$, which is a nonlinear functional of the full CDF -- not an expectation of the form $\mathbb{E}[p^k \cdot f(m)]$.

However, the product-measure approximation ($p \perp a$) gives a good CDF because $\text{Corr}(p, a) \approx -0.006$. We verify this by comparing the product-measure Lorenz against MC.

In [3]:
# Verify: product-measure Lorenz vs MC Lorenz
from HARK.utilities import get_lorenz_shares

# MC Lorenz: wealth = p * (1-S) * aNrm
lorenz_mc_list = []
for t in range(T_START, T):
    p_t = p_mc[t]; a_t = mc["aNrm_all"][t]
    W_t = p_t * (1 - Splurge) * a_t
    W_t = np.maximum(W_t, 0)
    lorenz_mc_list.append(get_lorenz_shares(W_t, percentiles=[0.2, 0.4, 0.6, 0.8]))
lorenz_mc = np.mean(lorenz_mc_list, axis=0) * 100

# Product-measure Lorenz: draw p independently, combine with TM a distribution
rng = np.random.default_rng(42)
N_synth = 50_000
a_flat = a_nrm.ravel()
erg_flat = erg.ravel()
idx = rng.choice(len(erg_flat), size=N_synth, p=erg_flat)
a_synth = a_flat[idx % M]
p_synth = rng.choice(p_grid, size=N_synth, p=p_w)
W_synth = p_synth * (1 - Splurge) * a_synth
W_synth = np.maximum(W_synth, 0)
lorenz_prod = np.array(get_lorenz_shares(W_synth, percentiles=[0.2, 0.4, 0.6, 0.8])) * 100

print(f"{'Percentile':>12s}  {'MC Lorenz':>10s}  {'Product':>10s}  {'Diff (pp)':>10s}")
for i, pct in enumerate([20, 40, 60, 80]):
    print(f"{pct:>12d}  {lorenz_mc[i]:10.2f}%  {lorenz_prod[i]:10.2f}%  {lorenz_prod[i]-lorenz_mc[i]:+10.2f}pp")
print(f"\nThe product-measure Lorenz is close to MC,")
print(f"confirming that the tiny p-a correlation has minimal impact on wealth ranking.")


  Percentile   MC Lorenz     Product   Diff (pp)
          20        0.65%        0.53%       -0.12pp
          40        6.28%        6.16%       -0.12pp
          60       19.02%       18.90%       -0.13pp
          80       42.77%       42.69%       -0.08pp

The product-measure Lorenz is close to MC,
confirming that the tiny p-a correlation has minimal impact on wealth ranking.


## 6 -- Summary

| Calculation | Type | Kernel? | Approach |
|------------|------|:-------:|----------|
| Consumption, income, NPV, multipliers, IRFs | $p$-linear | No | Harmenberg (exact) |
| **Marginal utility** $\mathbb{E}[u'(c)]$ | $p^{-\rho}$-separable | **Yes** | Kernel with $k{=}{-}\rho$ |
| **CRRA welfare** $\mathbb{E}[u(c)]$ | $p^{1-\rho}$-separable | **Yes** | Kernel with $k{=}1{-}\rho$ |
| **Policy welfare impact** $\mathbb{E}[\Delta u]$ | $p^{1-\rho}$-separable | **Yes** | Kernel (counterfactual $X$) |
| Lorenz curve, Gini, quantile shares | Distributional | **No** | Product-measure convolution (~1pp) |
| Check stimulus (income phase-out) | Non-separable | **No** | $p$-buckets |

**The general principle:** any $\mathbb{E}[p^k \cdot f(m, j, \theta)]$ admits a kernel -- change $k$ in the $\Phi^k$ factor, keep everything else the same. The cost is $O(M \times J^2 \times K_{\text{shocks}})$ regardless of $k$.

**What the kernel can't do:** distributional functionals (quantiles, Lorenz, Gini) and income-dependent policies where $p$ enters the consumption function's *argument*. For the former, the product-measure approximation works well; for the latter, $p$-buckets are needed.